# Statystyka dla Data Analyst/Engineer — Moduł 12: Regresja liniowa

W Module 11 nauczyliśmy się mierzyć, JAK SILNY jest związek między dwiema zmiennymi
(korelacja). **Regresja liniowa** idzie o krok dalej: buduje konkretny model
matematyczny, który pozwala PRZEWIDZIEĆ wartość jednej zmiennej na podstawie drugiej
(albo kilku innych) — z konkretnym równaniem, konkretnymi współczynnikami i miarą tego,
jak dobrze model pasuje do danych.

## Spis treści
1. [Prosta regresja liniowa](#sec1)
2. [R-kwadrat: miara dopasowania](#sec2)
3. [Diagnostyka reszt](#sec3)
4. [Regresja wieloraka](#sec4)
5. [R² skorygowane](#sec5)
6. [Predykcja nowych wartości](#sec6)
7. [Podsumowanie i ćwiczenia](#sec7)


Każdy moduł tego kursu zaczyna się od wygenerowania tego samego zestawu danych syntetycznych -- dzięki ustalonemu ziarnu losowości (`np.random.default_rng(42)`) liczby zawsze wyjdą identyczne, więc możesz otworzyć dowolny moduł niezależnie od pozostałych, nawet po restarcie kernela.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 400

miasto = rng.choice(
    ["Warszawa", "Krakow", "Gdansk", "Wroclaw", "Poznan"],
    size=n, p=[0.30, 0.25, 0.15, 0.15, 0.15],
)
region = rng.choice(["Polnoc", "Poludnie", "Wschod", "Zachod"], size=n)
wiek = rng.integers(18, 70, size=n)

# wydatki miesieczne: rozklad prawoskosny (typowy dla danych o wydatkach)
wydatki_miesieczne = rng.lognormal(mean=5.2, sigma=0.5, size=n).round(2)

# satysfakcja skorelowana z wydatkami + efekt regionu (celowo rozny) + szum losowy,
# przyciete do skali 1-10 -- dzieki temu Modul 10 (ANOVA) ma realna roznice miedzy grupami
efekt_regionu = {"Polnoc": 0.0, "Poludnie": -0.5, "Wschod": 0.3, "Zachod": 0.8}
satysfakcja = np.clip(
    wydatki_miesieczne / 45
    + np.array([efekt_regionu[r] for r in region])
    + rng.normal(0, 1.5, size=n),
    1, 10,
).round(1)

# grupa kampanii A/B -- z celowo wprowadzonym efektem w konwersji (grupa B lepsza)
grupa = rng.choice(["A", "B"], size=n)
prawdopodobienstwo_konwersji = np.where(grupa == "A", 0.10, 0.20)
konwersja = rng.binomial(1, prawdopodobienstwo_konwersji)

# wydatki przed/po wprowadzeniu programu lojalnosciowego (dane sparowane)
wydatki_przed = rng.normal(220, 40, size=n).round(2)
efekt_programu = rng.normal(15, 10, size=n)
wydatki_po = (wydatki_przed + efekt_programu).round(2)

klienci = pd.DataFrame({
    "klient_id": np.arange(1, n + 1),
    "miasto": miasto,
    "region": region,
    "wiek": wiek,
    "wydatki_miesieczne": wydatki_miesieczne,
    "satysfakcja": satysfakcja,
    "grupa_kampanii": grupa,
    "konwersja": konwersja,
    "wydatki_przed": wydatki_przed,
    "wydatki_po": wydatki_po,
})
klienci.head()


<a id="sec1"></a>
## 1. Prosta regresja liniowa

**Prosta regresja liniowa** (*simple linear regression*) szuka najlepiej dopasowanej
linii prostej `y = b0 + b1*x`, opisującej zależność zmiennej zależnej `y` (to, co
przewidujemy) od jednej zmiennej niezależnej `x` (predyktora). "Najlepiej dopasowanej"
oznacza: minimalizującej sumę kwadratów odległości punktów od linii (*metoda najmniejszych
kwadratów*, OLS — *Ordinary Least Squares*).

Sprawdźmy, jak `wydatki_miesieczne` (predyktor) przewidują `satysfakcja` (zmienna
zależna) — dokładnie tę parę, którą badaliśmy korelacyjnie w Module 11.

In [ ]:
from statsmodels.formula.api import ols

model = ols("satysfakcja ~ wydatki_miesieczne", data=klienci).fit()
print(model.params)


`Intercept` (`b0`) to przewidywana wartość `satysfakcja`, gdy `wydatki_miesieczne = 0`.
Współczynnik przy `wydatki_miesieczne` (`b1`, *nachylenie*, *slope*) mówi, o ile
zmienia się przewidywana `satysfakcja`, gdy `wydatki_miesieczne` rosną o jednostkę (1
zł). Pełne, szczegółowe podsumowanie modelu (błędy standardowe, wartości p, przedziały
ufności dla każdego współczynnika) dostajemy przez `.summary()`:

In [ ]:
print(model.summary())


> 📊 **Wartość p współczynnika to test istotności**
>
> Wiersz `wydatki_miesieczne` w tabeli ma swoją własną wartość `p` — to test H0: `b1 = 0` (predyktor nie ma żadnego liniowego wpływu na `y`). Bardzo mała wartość p oznacza, że nachylenie jest istotnie różne od zera — dokładnie ten sam rodzaj testu hipotez z Modułu 7, tylko zastosowany do współczynnika regresji.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(7, 5))
plt.scatter(klienci["wydatki_miesieczne"], klienci["satysfakcja"], alpha=0.4, label="dane")

x_linia = np.linspace(klienci["wydatki_miesieczne"].min(), klienci["wydatki_miesieczne"].max(), 100)
y_linia = model.params["Intercept"] + model.params["wydatki_miesieczne"] * x_linia
plt.plot(x_linia, y_linia, color="red", linewidth=2, label="linia regresji")

plt.xlabel("Wydatki miesieczne")
plt.ylabel("Satysfakcja")
plt.legend()
plt.show()


<a id="sec2"></a>
## 2. R-kwadrat: miara dopasowania

**R-kwadrat** (`R^2`, *coefficient of determination*) mówi, jaki PROCENT wariancji
zmiennej zależnej model wyjaśnia. `R^2 = 0` oznacza, że model nic nie wyjaśnia (tak samo
dobrze moglibyśmy zawsze przewidywać średnią), `R^2 = 1` oznacza idealne dopasowanie.

In [ ]:
print(f"R^2 = {model.rsquared:.4f}")


> 💡 **R² w prostej regresji to po prostu r² z Modułu 11**
>
> Dla regresji z JEDNYM predyktorem, `R^2` jest dokładnie kwadratem korelacji Pearsona między `x` i `y` z Modułu 11 (`r^2`). To nie przypadek — obie miary opierają się na tej samej idei "jak dobrze liniowa zależność opisuje dane", tylko patrzą na nią z dwóch różnych stron.

In [ ]:
from scipy import stats

r, _ = stats.pearsonr(klienci["wydatki_miesieczne"], klienci["satysfakcja"])
print(f"r^2 z korelacji Pearsona: {r ** 2:.4f}")
print(f"R^2 z regresji: {model.rsquared:.4f}")


<a id="sec3"></a>
## 3. Diagnostyka reszt

**Reszty** (*residuals*) to różnice między wartościami rzeczywistymi a przewidzianymi
przez model (`reszta = y_rzeczywiste - y_przewidziane`). Dobra regresja liniowa zakłada,
że reszty są w przybliżeniu normalne i mają stałą wariancję niezależnie od
przewidywanej wartości (*homoscedastyczność*) — brak tego nazywa się
*heteroscedastycznością*.

In [ ]:
reszty = model.resid
przewidywane = model.fittedvalues

plt.figure(figsize=(7, 5))
plt.scatter(przewidywane, reszty, alpha=0.4)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Wartosci przewidywane")
plt.ylabel("Reszty")
plt.title("Wykres reszt vs wartosci przewidywane")
plt.show()


Chmura punktów bez wyraźnego wzorca (rozrzucona losowo wokół linii `0`) sugeruje, że
założenie homoscedastyczności jest w porządku. Lejek albo wyraźna krzywizna
sugerowałyby problem. Sprawdźmy też normalność reszt testem Shapiro-Wilka (Moduł 8):

In [ ]:
statystyka_sw, wartosc_p_sw = stats.shapiro(reszty)
print(f"Shapiro-Wilk: statystyka={statystyka_sw:.4f}, p={wartosc_p_sw:.4f}")


> ⚠️ **Formalna nieistotność normalności nie zawsze jest problemem**
>
> Przy `n=400` wartość p testu Shapiro-Wilka może wyjść nieco poniżej `0.05` (u nas tak właśnie jest) — częściowo dlatego, że `satysfakcja` w naszych danych była sztucznie przycięta (`clip`) do przedziału `[1, 10]`, co lekko zaburza normalność w ogonach rozkładu. Testy statystyczne przy dużej próbie wykrywają nawet BARDZO drobne odchylenia od idealnej normalności (dokładnie ten sam problem, co z małymi efektami w Module 7). W praktyce najpierw patrzy się na wykres reszt (histogram albo wykres kwantyl-kwantyl), a dopiero potem na formalny test.

In [ ]:
import statsmodels.api as sm

sm.qqplot(reszty, line="45", fit=True)
plt.title("Wykres kwantyl-kwantyl (Q-Q) reszt")
plt.show()


<a id="sec4"></a>
## 4. Regresja wieloraka

**Regresja wieloraka** (*multiple linear regression*) rozszerza ten sam pomysł na WIELE
predyktorów naraz: `y = b0 + b1*x1 + b2*x2 + ...`. Dodajmy `region` (zmienną kategoryczną
— pamiętaj o `C(...)` z Modułu 10!) do modelu z sekcji 1:

In [ ]:
model_wieloraki = ols("satysfakcja ~ wydatki_miesieczne + C(region)", data=klienci).fit()
print(model_wieloraki.params)
print(f"R^2 = {model_wieloraki.rsquared:.4f}")


`R^2` wzrósł względem modelu z jednym predyktorem (sekcja 2) — region rzeczywiście
niesie dodatkową informację, dokładnie tak, jak sugerował box plot z Modułu 2 i ANOVA z
Modułu 10. Współczynniki przy poszczególnych regionach (`C(region)[T.Poludnie]` itd.)
pokazują różnicę względem kategorii bazowej (tej, która nie ma własnego wiersza —
`statsmodels` wybiera ją automatycznie w kolejności alfabetycznej) — i z grubsza
odzwierciedlają ten sam wzorzec różnic między regionami, który sami zbudowaliśmy w danych
w Module 8/10 (Zachód najwyżej, Południe najniżej).

<a id="sec5"></a>
## 5. R² skorygowane

Zwykłe `R^2` ma wadę: rośnie (albo w najgorszym razie zostaje bez zmian) za KAŻDYM
razem, gdy dodamy nowy predyktor — nawet zupełnie bezużyteczny, losowy. **R² skorygowane**
(*adjusted R²*) karze za dodawanie predyktorów, które nie poprawiają realnie modelu, więc
lepiej nadaje się do porównywania modeli o różnej liczbie zmiennych.

In [ ]:
print(f"R^2 (wydatki + region): {model_wieloraki.rsquared:.4f}")
print(f"R^2 skorygowane (wydatki + region): {model_wieloraki.rsquared_adj:.4f}")


> 📊 **Jeśli R² skorygowane spada, predyktor prawdopodobnie nie pomaga**
>
> Dodanie predyktora, który naprawdę nic nie wnosi (np. czystego szumu), zwykle sprawia, że zwykłe `R^2` rośnie o mikroskopijny ułamek, a `R^2` skorygowane wręcz SPADA — bo kara za dodatkowy parametr przewyższa znikomy zysk w dopasowaniu. Zobaczysz to na własne oczy w ćwiczeniu 2 poniżej.

<a id="sec6"></a>
## 6. Predykcja nowych wartości

Mając dopasowany model, możemy przewidzieć `satysfakcja` dla zupełnie NOWEGO klienta,
którego jeszcze nie widzieliśmy — wystarczy podać wartości predyktorów w tym samym
formacie co dane treningowe:

In [ ]:
nowy_klient = pd.DataFrame({"wydatki_miesieczne": [200.0], "region": ["Zachod"]})
przewidywana_satysfakcja = model_wieloraki.predict(nowy_klient)
print(f"Przewidywana satysfakcja: {przewidywana_satysfakcja.values[0]:.2f}")


> ⚠️ **Ekstrapolacja poza zakres danych treningowych jest ryzykowna**
>
> Model "zna" tylko zakres wartości, na których był trenowany. Przewidywanie dla `wydatki_miesieczne = 100000` (dużo poza zakresem naszych danych) może dać absurdalny wynik — linia prosta, dopasowana lokalnie, nie musi w ogóle pasować do zachowania danych daleko poza obserwowanym zakresem.

<a id="sec7"></a>
## 7. Podsumowanie i ćwiczenia

W tym module poznaliśmy:
- prostą regresję liniową (`statsmodels.formula.api.ols`) i interpretację współczynników
  (`Intercept`, nachylenie),
- `R^2` jako miarę dopasowania, i jego związek z korelacją Pearsona z Modułu 11,
- diagnostykę reszt: wykres reszt vs wartości przewidywane (homoscedastyczność), test
  Shapiro-Wilka i wykres Q-Q (normalność),
- regresję wieloraką z predyktorami kategorycznymi (`C(...)`),
- `R^2` skorygowane jako lepszą miarę do porównywania modeli o różnej liczbie
  predyktorów,
- predykcję dla nowych obserwacji przez `.predict()`.

> 📝 **Ćwiczenie 1: Regresja wydatków po programie na wydatkach przed programem**
>
> Zbuduj prostą regresję `wydatki_po ~ wydatki_przed` (dane z Modułu 8). Wypisz współczynniki i `R^2`. Czy nachylenie jest bliskie `1`? Co to sugeruje o zależności między tymi dwiema zmiennymi?

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
model_przed_po = ols("wydatki_po ~ wydatki_przed", data=klienci).fit()
print(model_przed_po.params)
print(f"R^2 = {model_przed_po.rsquared:.4f}")
```

Nachylenie wychodzi bliskie 1, a R^2 bardzo wysokie (ok. 0.95) -- to spodziewany wynik, bo w Module 8 skonstruowalismy wydatki_po jako wydatki_przed plus losowy efekt o sredniej ok. 15. Intercept jest rzedu tej wartosci (choc nie identyczny - to tylko dopasowanie do konkretnej probki), co potwierdza ta konstrukcje.
</details>

> 📝 **Ćwiczenie 2: Czy wiek poprawia model?**
>
> Zbuduj model `satysfakcja ~ wydatki_miesieczne + wiek` i porównaj jego `R^2` oraz `R^2` skorygowane z modelem z sekcji 1 (samo `wydatki_miesieczne`). Sprawdź też wartość p współczynnika przy `wiek`. Czy `wiek` realnie poprawia model?

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
model_z_wiekiem = ols("satysfakcja ~ wydatki_miesieczne + wiek", data=klienci).fit()
print(f"R^2 (bez wieku): {model.rsquared:.4f}, skorygowane: {model.rsquared_adj:.4f}")
print(f"R^2 (z wiekiem): {model_z_wiekiem.rsquared:.4f}, skorygowane: {model_z_wiekiem.rsquared_adj:.4f}")
print(f"Wartosc p dla wiek: {model_z_wiekiem.pvalues['wiek']:.4f}")
```

Zwykle R^2 rosnie o mikroskopijny ulamek, ale R^2 SKORYGOWANE lekko SPADA, a wartosc p dla wiek jest wysoko powyzej 0.05 -- dokladnie ten mechanizm z sekcji 5. Wiek jest w naszych danych losowany calkowicie niezaleznie od reszty, wiec nie wnosi realnej sily predykcyjnej.
</details>

> 📝 **Ćwiczenie 3: Diagnostyka reszt dla modelu z regionem**
>
> Narysuj wykres reszt vs wartości przewidywane dla `model_wieloraki` (sekcja 4, z regionem). Czy widzisz jakiś niepokojący wzorzec (np. lejek)?

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
reszty_wieloraki = model_wieloraki.resid
przewidywane_wieloraki = model_wieloraki.fittedvalues

plt.figure(figsize=(7, 5))
plt.scatter(przewidywane_wieloraki, reszty_wieloraki, alpha=0.4)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Wartosci przewidywane")
plt.ylabel("Reszty")
plt.show()
```
</details>

> 📝 **Ćwiczenie 4: Predykcja dla kilku nowych klientów naraz**
>
> Użyj `model_wieloraki.predict()`, żeby przewidzieć `satysfakcja` dla TRZECH nowych klientów naraz (różne kombinacje `wydatki_miesieczne` i `region`), podając wszystkie na raz w jednym DataFrame.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
nowi_klienci = pd.DataFrame({
    "wydatki_miesieczne": [100.0, 250.0, 400.0],
    "region": ["Polnoc", "Poludnie", "Zachod"],
})
model_wieloraki.predict(nowi_klienci)
```
</details>

> 🔥 **Wyzwanie: czy grupa kampanii pomaga, gdy kontrolujemy wydatki?**
>
> Zbuduj model `satysfakcja ~ wydatki_miesieczne + C(region) + C(grupa_kampanii)` -- pełny model z trzema predyktorami. Sprawdź wartość p współczynnika przy `grupa_kampanii`. Porównaj wynik z dwuczynnikową ANOVA z Modułu 10 (Wyzwanie), która NIE uwzględniała wydatków -- czy wniosek o istotności `grupa_kampanii` się zmienił?

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
model_pelny = ols(
    "satysfakcja ~ wydatki_miesieczne + C(region) + C(grupa_kampanii)", data=klienci
).fit()
print(f"Wartosc p dla grupa_kampanii: {model_pelny.pvalues.filter(like='grupa_kampanii').iloc[0]:.4f}")
print(f"R^2: {model_pelny.rsquared:.4f} (vs {model_wieloraki.rsquared:.4f} bez grupy kampanii)")
```

Tutaj (z wydatkami jako dodatkowym predyktorem) grupa_kampanii NIE wychodzi istotna (p wyraznie powyzej 0.05) -- inaczej niz w dwuczynnikowej ANOVA z Modulu 10, gdzie (bez wydatkow w modelu) wyszla istotna na poziomie p=0.04 jako falszywy alarm (blad I rodzaju). To dobra ilustracja, ze wniosek o istotnosci zmiennej moze zalezec od tego, jakie INNE zmienne uwzglednimy w modelu -- dodanie wydatkow jako kontrolnego predyktora tutaj 'wchlonelo' czesc szumu, ktory wczesniej przypadkowo trafil na grupe kampanii.
</details>

## Co dalej?

Regresja liniowa przewiduje zmienną CIĄGŁĄ (np. satysfakcję w skali 1-10). Ale co, gdy
chcemy przewidzieć coś BINARNEGO — np. czy klient dokona konwersji (tak/nie)? W **Module
13** poznamy **regresję logistyczną**, zaprojektowaną dokładnie do tego celu.